# HW14. Подбор гиперпараметров и интерпретация моделей

В этой работе я на датасете Breast Cancer Wisconsin сравниваю базовые модели, подбираю гиперпараметры для SVM и Random Forest, а потом смотрю, какие признаки реально влияют на качество модели.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Данные

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

display(X.head())
print("Размер данных:", X.shape)
print("Классы:", dict(zip(data.target_names, np.bincount(y))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

## 2. Базовые модели

Сначала беру несколько моделей без специального подбора параметров, чтобы было с чем сравнивать улучшения.

In [ ]:
base_models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(random_state=RANDOM_STATE)),
    ]),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
}

baseline_rows = []
for name, model in base_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1_macro")
    baseline_rows.append({
        "model": name,
        "cv_f1_mean": scores.mean(),
        "cv_f1_std": scores.std(),
    })

baseline_results = pd.DataFrame(baseline_rows).sort_values("cv_f1_mean", ascending=False)
baseline_results

## 3. Grid Search для SVM

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(probability=True, random_state=RANDOM_STATE)),
])

svm_param_grid = {
    "svm__kernel": ["rbf", "linear"],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.001],
}

svm_grid = GridSearchCV(
    svm_pipeline,
    svm_param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
)
svm_grid.fit(X_train, y_train)

print("Лучшие параметры:", svm_grid.best_params_)
print(f"Лучший CV F1_macro: {svm_grid.best_score_:.4f}")

svm_grid_results = (
    pd.DataFrame(svm_grid.cv_results_)
    [["params", "mean_test_score", "std_test_score", "rank_test_score"]]
    .sort_values("rank_test_score")
)
svm_grid_results.head(10)

## 4. Random Search для Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=RANDOM_STATE)

rf_param_dist = {
    "n_estimators": [100, 200, 400, 600],
    "max_depth": [None, 3, 5, 8, 12],
    "min_samples_split": [2, 4, 8, 12],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

rf_random = RandomizedSearchCV(
    rf,
    rf_param_dist,
    n_iter=35,
    cv=5,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_random.fit(X_train, y_train)

print("Лучшие параметры:", rf_random.best_params_)
print(f"Лучший CV F1_macro: {rf_random.best_score_:.4f}")

## 5. Финальное сравнение на тестовой выборке

In [ ]:
candidates = {
    "SVM GridSearch": svm_grid.best_estimator_,
    "RandomForest RandomSearch": rf_random.best_estimator_,
}
candidates.update(base_models)

test_rows = []
for name, model in candidates.items():
    fitted = model.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    test_rows.append({"model": name, "test_f1_macro": f1_score(y_test, y_pred, average="macro")})

test_results = pd.DataFrame(test_rows).sort_values("test_f1_macro", ascending=False)
display(test_results)

best_name = test_results.iloc[0]["model"]
best_model = candidates[best_name].fit(X_train, y_train)
print("Лучшая модель на тесте:", best_name)
print(classification_report(y_test, best_model.predict(X_test), target_names=data.target_names))

## 6. Permutation Importance

Permutation importance показывает, насколько падает качество, если перемешать отдельный признак. Это полезнее простого `feature_importances_`, потому что оценка считается на отложенной выборке.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)
display(importance.head(12))

plt.figure(figsize=(8, 5))
top = importance.head(10).iloc[::-1]
plt.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
plt.xlabel("Падение F1_macro после перемешивания")
plt.title("Permutation importance: топ признаков")
plt.tight_layout()
plt.show()

## 7. PDP / ICE

На графиках ниже видно, как меняется предсказание модели при изменении самых важных признаков.

In [ ]:
top_features = importance.head(3)["feature"].tolist()
fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    best_model,
    X_train,
    features=top_features,
    kind="both",
    subsample=60,
    random_state=RANDOM_STATE,
    ax=ax,
)
plt.tight_layout()
plt.show()

## 8. SHAP, если библиотека установлена

In [ ]:
try:
    import shap

    rf_for_shap = rf_random.best_estimator_
    explainer = shap.TreeExplainer(rf_for_shap)
    shap_values = explainer.shap_values(X_test)
    values_for_positive_class = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(values_for_positive_class, X_test, max_display=10, show=True)
except Exception as exc:
    print("SHAP-график пропущен:", exc)

## Вывод

Подбор параметров обычно даёт небольшой, но заметный прирост относительно совсем базовых моделей. Самыми полезными признаками оказываются характеристики размера и формы клеточных ядер. При этом важности стоит читать осторожно: похожие между собой признаки могут делить вклад, поэтому один конкретный столбец не всегда показывает всю медицинскую картину.